# Prediction with missing limbs and joints

In practice, not all joints of a skeleton are always observed (e.g. occlusions or a missing sensor).
Since EquiFusion treats the skeleton as a graph and is permutation equivariant, missing parts can simply be **removed from the graph**, and the model predicts the motion of the remaining ones.

This notebook shows three settings:
1. **Bone direction model, missing limbs**: limbs are missing (here the left arm and the right leg), and the prediction does not contain them.
2. **3D joint model, missing joints**: arbitrary joints (e.g. a knee and an elbow) are missing.
3. **Bone direction model, generating the missing limbs**: the motion of the missing limbs is also generated.

**Which parts can be missing for the bone direction model?**
It predicts bone directions, and joint positions are recovered by integrating the (unit) bone directions along the kinematic chain, from the root outwards. The missing parts must therefore not separate the remaining skeleton into disconnected chains: removing a joint in the middle of a chain would disconnect every joint after it, whose position could not be recovered. The limbs `head`, `left_arm`, `right_arm`, `left_leg` and `right_leg` end at an extremity, so any combination of them can be missing.
The 3D joint model predicts the position of each joint directly, so any joint except the root can be missing.

In [1]:
import numpy as np
import torch
from src.inference_utils import load_equifusion, predict
from src.metrics.ranking import get_closest_and_nfurthest_maxapd

# Released models, expected in ./trained_models; they can also be downloaded from Hugging Face, e.g.
# checkpoint_path = download_equifusion('EquiFusion/equifusion-cross-kinematics-amass-nymeria')  (from src.inference_utils)
ckpt = lambda name: f'./trained_models/cross_kinematics/{name}/checkpoints/checkpoint_375.safetensors'
num_samples = 50

obs = np.load('figures/examples/amass_missing_limbs_obs.npy')  # [30, 22, 3], AMASS test sequence (root joint included, at the origin)
gt = np.load('figures/examples/amass_missing_limbs_gt.npy')    # [120, 22, 3]
gt_rel = torch.from_numpy(gt[:, 1:] - gt[:, :1])

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML


def with_root(x):
    """[..., J-1, 3] predictions (relative to the root) -> [..., J, 3] with the root at the origin."""
    return np.concatenate([np.zeros_like(x[..., :1, :]), x], axis=-2)


def animate(obs, futures, titles, limbseqs, obs_limbseq, colors=None, step=2, fps=30):
    """Skeleton animation: the observed motion (gray) followed by one or more futures, one panel each.
    obs [T_obs, J, 3]; futures: list of [T_pred, J, 3], all root-relative with the root joint included (NaN = not drawn);
    limbseqs: list of bone lists (one per future); obs_limbseq: bones for the observed motion."""
    n, T_obs, T_pred = len(futures), len(obs), len(futures[0])
    colors = colors or ["tab:green"] + ["tab:red"] + ["tab:blue"] * n
    obs = obs - obs[:, :1]
    fig = plt.figure(figsize=(3 * n, 3.4), dpi=80)
    axes, lines = [], []
    for i in range(n):
        ax = fig.add_subplot(1, n, i + 1, projection="3d")
        ax.set(xlim=(-1, 1), ylim=(-1, 1), zlim=(-1, 1), title=titles[i])
        ax.set_box_aspect((1, 1, 1), zoom=1.35); ax.view_init(elev=10, azim=-60); ax.set_axis_off()
        lines.append([ax.plot([], [], [], lw=2)[0] for _ in range(max(len(limbseqs[i]), len(obs_limbseq)))])
    fig.subplots_adjust(left=0, right=1, bottom=0, top=0.9, wspace=0)

    def draw(t):
        for i in range(n):
            pose, bones, color = (obs[t], obs_limbseq, "gray") if t < T_obs else (futures[i][t - T_obs], limbseqs[i], colors[i])
            for k, line in enumerate(lines[i]):
                if k < len(bones):
                    a, b = bones[k]
                    line.set_data_3d(pose[[a, b], 0], pose[[a, b], 1], pose[[a, b], 2])
                else:
                    line.set_data_3d([], [], [])
                line.set_color(color)
        return sum(lines, [])

    anim = animation.FuncAnimation(fig, draw, frames=range(0, T_obs + T_pred, step), interval=1000 / fps)
    html = HTML(anim.to_html5_video())
    plt.close(fig)
    return html


def rank(pred, gt_rel, nsamples=2):
    """closest sample to the ground truth and `nsamples` diverse ones; NaN entries (missing joints) are ignored."""
    mask = torch.isnan(pred[0, 0]).any(-1)  # missing joints [J-1]
    gt_rel, pred = gt_rel.clone(), pred.clone()
    gt_rel[:, mask], pred[:, :, mask] = 0, 0
    closest, furthest, _ = get_closest_and_nfurthest_maxapd(pred, gt_rel, nsamples=nsamples)
    selected = torch.cat([closest.unsqueeze(0), furthest])
    selected[..., mask, :] = float('nan')  # missing joints are not drawn
    return selected.numpy()


## 1. Bone direction model: missing limb
We use the bone direction model `cross_kinematics/amass_nymeria` and declare the left arm and the right leg as missing. Their bones are removed from the graph, and the model predicts the rest of the body. The joints of the missing limbs are not predicted: they are NaN in the output, which has the joint order of the dataset.

`missing_limbs` accepts the limb names and also the names of single bones (printed below), e.g. `['LLowerArm']` for a missing left forearm. Every bone after a missing one in its kinematic chain must be missing too (e.g. `LUpperArm` only together with `LLowerArm`); otherwise an error tells which bones to add.

In [3]:
from src.inference_utils import LIMB_NAMES
model, skeleton, config = load_equifusion(ckpt('amass_nymeria'), 'amass', missing_limbs=['left_arm', 'right_leg'], num_samples=num_samples)

print('Limbs:', list(LIMB_NAMES))
print('Bones:', skeleton.bone_names)

missing = skeleton.drop_joints_ids_orig  # joint indices (root = 0) of the missing joints
print('Missing joints:', [skeleton.joint_dict_orig[j] for j in missing])

obs_missing = obs.copy()
obs_missing[:, missing] = np.nan  # not observed (ignored by the model)
pred = predict(model, skeleton, config, obs_missing, num_samples=num_samples, seed=0).cpu()  # [50, 120, 21, 3], NaN = missing
ade = torch.nanmean((pred - gt_rel).norm(dim=-1), dim=(-1, -2))
print(f'{(~torch.isnan(pred[0, 0, :, 0])).sum()} joints predicted. minADE on these joints: {ade.min():.3f} m')

Created encoder with 2.023M and decoder with 3.471M trainable parameters.
Model size Autoencoder (MB):  20.9599609375


Diffusion is_ddim_sampling:  False
Model size Diffusion (MB):  9.9093017578125
Limbs: ['head', 'left_arm', 'right_arm', 'left_leg', 'right_leg']
Bones: ['Spine1', 'Spine2', 'Spine3', 'Neck', 'Head', 'RCollarbone', 'RShoulder', 'RUpperArm', 'RLowerArm', 'LCollarbone', 'LShoulder', 'LUpperArm', 'LLowerArm', 'RHip', 'RThigh', 'RCalf', 'RFoot', 'LHip', 'LThigh', 'LCalf', 'LFoot']
Missing joints: ['RHeel', 'RFoot', 'LElbow', 'LWrist']


17 joints predicted. minADE on these joints: 0.108 m


In [4]:
from src.data import create_skeleton
from src.inference_utils import quick_cfg_for_inference_no_hydra
full_limbseq = create_skeleton(**quick_cfg_for_inference_no_hydra(ckpt('amass_nymeria'), 'amass')).limbseq  # skeleton without missing limbs

selected = rank(pred, gt_rel)
animate(obs_missing, [gt - gt[:, :1], *with_root(selected)],
        titles=['Ground truth', 'Closest to GT', 'Diverse 1', 'Diverse 2'],
        limbseqs=[full_limbseq] * 4, obs_limbseq=full_limbseq)

## 2. 3D joint model: any joint can be missing
The 3D joint model `cross_kinematics/amass_nymeria_3d` predicts joint positions directly, so also joints in the middle of a chain can be missing, for example a knee and an elbow (the heel, foot, wrist etc. are still observed). Missing joints can be given by name or by index, and are returned as NaN in the prediction.

In [5]:
model3d, skeleton3d, config3d = load_equifusion(ckpt('amass_nymeria_3d'), 'amass', num_samples=num_samples)
print('Joint names:', list(skeleton3d.node_dict.values()))

missing_joints = ['LKnee', 'LElbow']
missing3d = [j for j, name in skeleton3d.joint_dict_orig.items() if name in missing_joints]
obs_missing3d = obs.copy()
obs_missing3d[:, missing3d] = np.nan  # not observed (ignored by the model)

pred3d = predict(model3d, skeleton3d, config3d, obs_missing3d, num_samples=num_samples, seed=0, missing_joints=missing_joints).cpu()
print(pred3d.shape, '| NaN joints in the prediction:', torch.isnan(pred3d[0, 0, :, 0]).nonzero().flatten().tolist())
ade = torch.nanmean((pred3d - gt_rel).norm(dim=-1), dim=(-1, -2))
print(f'minADE on the predicted joints: {ade.min():.3f} m')

selected = rank(pred3d, gt_rel)
animate(obs_missing3d, [gt - gt[:, :1], *with_root(selected)],
        titles=['Ground truth', 'Closest to GT', 'Diverse 1', 'Diverse 2'],
        limbseqs=[skeleton3d.limbseq] * 4, obs_limbseq=skeleton3d.limbseq)

Created encoder with 2.023M and decoder with 3.471M trainable parameters.
Model size Autoencoder (MB):  20.9599609375
Diffusion is_ddim_sampling:  False
Model size Diffusion (MB):  9.9093017578125
Joint names: ['LHip', 'RHip', 'Spine1', 'LKnee', 'RKnee', 'Spine3', 'LHeel', 'RHeel', 'Neck', 'LFoot', 'RFoot', 'BMN', 'LSI', 'RSI', 'Head', 'LShoulder', 'RShoulder', 'LElbow', 'RElbow', 'LWrist', 'RWrist']


torch.Size([50, 120, 21, 3]) | NaN joints in the prediction: [3, 17]
minADE on the predicted joints: 0.145 m


## 3. Generating the missing limbs
With `generate_missing_limbs=True`, the bone direction model also generates the motion of the missing limb. We use a simple heuristic: the latents of the mirrored limb (here, the right arm) are used as conditioning for the missing one.
The output contains all joints, so the missing limb is generated.

In [6]:
model_gen, skeleton_gen, config_gen = load_equifusion(ckpt('amass_nymeria'), 'amass', missing_limbs=['left_arm'],
                                                      generate_missing_limbs=True, num_samples=num_samples)
obs_missing_arm = obs.copy()
missing_arm = skeleton_gen.drop_joints_ids_orig  # joints of the left arm
obs_missing_arm[:, missing_arm] = np.nan
pred_gen = predict(model_gen, skeleton_gen, config_gen, obs_missing_arm, num_samples=num_samples, seed=0).cpu()  # [50, 120, 21, 3]

miss = [j - 1 for j in missing_arm]  # missing joints in the prediction
err = (pred_gen - gt_rel).norm(dim=-1)
print(f'minADE all joints: {err.mean(dim=(-1, -2)).min():.3f} m | generated left arm only: {err[..., miss].mean(dim=(-1, -2)).min():.3f} m')

selected = rank(pred_gen, gt_rel)
animate(obs_missing_arm, [gt - gt[:, :1], *with_root(selected)],
        titles=['Ground truth', 'Closest to GT', 'Diverse 1', 'Diverse 2'],
        limbseqs=[full_limbseq] * 4, obs_limbseq=full_limbseq)

Created encoder with 2.023M and decoder with 3.471M trainable parameters.
Model size Autoencoder (MB):  20.9599609375
Diffusion is_ddim_sampling:  False
Model size Diffusion (MB):  9.9093017578125


minADE all joints: 0.123 m | generated left arm only: 0.260 m
